# IEEE 118: step-time comparison of conventional and SSN networks

This notebook compares the IEEE 118-bus simulations using conventional components and the passive SSN network. It separates component work **before** and **after** the network solve, compares total simulation step time, and estimates time remaining after pre-step and post-step work is removed.

## Main results

- The SSN network reduces measured pre-step and post-step work, especially for lines.
- In the detailed profile, the mean whole-step time is about **5.36 to 2.19 ms** in DP3Ph and **5.17 to 1.97 ms** in EMT3Ph.
- After subtracting all measured component pre-step and post-step task averages, the remainder is about **1.70 to 1.00 ms** in DP3Ph and **1.47 to 0.94 ms** in EMT3Ph.

The remainder includes the MNA solve, scheduler, other step-loop work, and profiling overhead. It is not an isolated timer for the numerical integration formula.

## Simulation setup and timing definitions

Both variants use the same IEEE 118-bus topology and 54 fourth-order VBR synchronous generators. The SSN variant replaces pi-lines, loads and shunts, and transformers with their SSN counterparts.

| Setting | DP3Ph | EMT3Ph |
|---|---:|---:|
| Simulated duration | 1 s | 1 s |
| Time step | 1 ms | 0.5 ms |
| Steps | 1,000 | 2,000 |
| Solver | MNA + KLU | MNA + KLU |
| Data logger | Off | Off |
| Profile instrumentation | Task, solver, and step timing | Task, solver, and step timing |

**Pre-step** and **post-step** plots sum the measured average time for each task group over one simulation step. **Whole-step** is wall time around the complete scheduler step. **After pre/post** is whole-step wall time minus the summed pre-step and post-step averages. Because instrumentation boundaries differ, treat the last quantity as an approximate remainder.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the checked-in data whether launched from the repository root or this folder.
rel = Path("examples/Notebooks/Performance/data/IEEE118_Component_Task_Profile")
local = Path.cwd() / "data/IEEE118_Component_Task_Profile"
if local.is_dir():
    data_dir = local
else:
    root = next(
        (p for p in [Path.cwd(), *Path.cwd().parents] if (p / rel).is_dir()), None
    )
    if root is None:
        raise FileNotFoundError(f"Could not find {rel}")
    data_dir = root / rel

tasks = pd.read_csv(data_dir / "task_times.csv")
steps = pd.read_csv(data_dir / "step_times.csv")
profile_runs = pd.read_csv(data_dir / "profile_runs.csv")
controls = pd.read_csv(data_dir / "endpoint_controls.csv")

case_names = {"conventional": "Conventional", "ssn_network": "SSN network"}
case_order = ["Conventional", "SSN network"]
colors = {"Conventional": "#4C78A8", "SSN network": "#F28E2B"}
phase_names = {"MnaPreStep": "Pre-step", "MnaPostStep": "Post-step"}
component_order = ["Lines", "Loads + shunts", "Transformers", "Generators"]
plt.rcParams.update({"font.size": 10, "axes.titlesize": 12, "axes.labelsize": 10})

tasks["case_label"] = tasks["case"].map(case_names)
tasks["task_ms"] = tasks["average_ns"] / 1e6
# Total mean cost in each component group, phase, domain, and case per simulated step.
groups = (
    tasks[tasks.sample_count > 0]
    .groupby(
        ["domain", "case", "case_label", "component_group", "phase"], as_index=False
    )
    .agg(ms_per_step=("task_ms", "sum"))
)


def comparison_legend(fig):
    # Fixed full-width legend frame, matching font, spacing, and placement in every chart.
    fig.legend(
        case_order,
        loc="lower left",
        bbox_to_anchor=(0, 1.005, 1, 0.065),
        mode="expand",
        ncol=2,
        frameon=False,
        fontsize=9,
        borderaxespad=0,
        handlelength=1.8,
        labelspacing=0.5,
        columnspacing=1.5,
    )


def plot_phase(phase, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=False)
    for ax, domain in zip(axes, ["DP3Ph", "EMT3Ph"]):
        sub = groups[(groups.domain == domain) & (groups.phase == phase)]
        pivot = sub.pivot_table(
            index="component_group",
            columns="case_label",
            values="ms_per_step",
            fill_value=0,
        )
        pivot = pivot.reindex(component_order).fillna(0).reindex(columns=case_order)
        pivot.plot(kind="bar", ax=ax, color=[colors[c] for c in case_order], width=0.76)
        ax.set_title(domain)
        ax.set_xlabel("")
        ax.set_ylabel("Sum of task averages (ms/step)")
        ax.tick_params(axis="x", rotation=18)
        ax.grid(axis="y", alpha=0.22)
        ax.set_axisbelow(True)
        ax.get_legend().remove()
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
    comparison_legend(fig)
    fig.suptitle(title, y=1.12, fontsize=14)
    fig.tight_layout()
    plt.show()

## 1. Pre-step comparison

This plot compares the component updates scheduled before the network solve. Each bar is the sum of the average task time for that component group per simulation step.

In [ ]:
plot_phase("MnaPreStep", "Component pre-step time by group")

## 2. Post-step comparison

This plot compares the component state and current updates scheduled after the network solve. The categories, colors, units, and legend format match the pre-step plot.

In [ ]:
plot_phase("MnaPostStep", "Component post-step time by group")

## 3. Whole simulation-step comparison

The whole-step wall timer includes component pre/post tasks, the solve, scheduling, and the rest of the simulation loop. It is measured with per-task profiling enabled, so these absolute values are higher than the no-logger, unprofiled endpoint controls shown below.

In [ ]:
whole = profile_runs.groupby(["domain", "case"], as_index=False).agg(
    mean_step_ms=("mean_step_s", "mean")
)
whole["mean_step_ms"] *= 1000  # Convert the saved mean from seconds to milliseconds.
whole["case_label"] = whole["case"].map(case_names)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=False)
for ax, domain in zip(axes, ["DP3Ph", "EMT3Ph"]):
    sub = whole[whole.domain == domain].set_index("case_label").loc[case_order]
    bars = ax.bar(
        case_order, sub.mean_step_ms, color=[colors[c] for c in case_order], width=0.62
    )
    ax.bar_label(bars, fmt="%.2f ms", padding=3, fontsize=9)
    ax.set_title(domain)
    ax.set_ylabel("Mean whole-step wall time (ms/step)")
    ax.grid(axis="y", alpha=0.22)
    ax.set_axisbelow(True)
fig.suptitle("Whole simulation-step time (profiled runs)", y=1.12, fontsize=14)
fig.tight_layout()
plt.show()

# Report the separate unprofiled current-build endpoint controls as a table.
current = controls[controls.measurement == "current_build_no_logger"].copy()
current["case_label"] = current["case"].map(case_names)
current["mean_step_ms"] = (
    current["calculation_s"]
    / current["simulated_seconds"]
    * current["timestep_s"]
    * 1000
)
print("Separate current-build controls (unprofiled, data logger off):")
display(current[["domain", "case_label", "mean_step_ms"]].round(3))

## 4. Step time excluding pre-step and post-step

The chart below subtracts the sum of all component `MnaPreStep` and `MnaPostStep` task averages from the profiled whole-step wall time. The remainder includes the MNA solve and scheduler/loop/profiling overhead. It is the closest available estimate of time outside component pre/post updates, not a pure integration-kernel timer.

In [ ]:
prepost = (
    groups[groups.phase.isin(["MnaPreStep", "MnaPostStep"])]
    .groupby(["domain", "case", "case_label"], as_index=False)
    .agg(prepost_ms=("ms_per_step", "sum"))
)
step_split = whole.merge(prepost, on=["domain", "case", "case_label"], how="left")
step_split["remaining_ms"] = step_split["mean_step_ms"] - step_split["prepost_ms"]
display(
    step_split[
        ["domain", "case_label", "prepost_ms", "remaining_ms", "mean_step_ms"]
    ].round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=False)
for ax, domain in zip(axes, ["DP3Ph", "EMT3Ph"]):
    sub = (
        step_split[step_split.domain == domain].set_index("case_label").loc[case_order]
    )
    bars = ax.bar(
        case_order, sub.remaining_ms, color=[colors[c] for c in case_order], width=0.62
    )
    ax.bar_label(bars, fmt="%.2f ms", padding=3, fontsize=9)
    ax.set_title(domain)
    ax.set_ylabel("Remaining time after pre/post (ms/step)")
    ax.grid(axis="y", alpha=0.22)
    ax.set_axisbelow(True)
fig.suptitle(
    "Step time after removing component pre-step and post-step", y=1.12, fontsize=14
)
fig.tight_layout()
plt.show()

## Interpretation and limitations

The largest savings are in line pre-step and post-step work. Loads/shunts also contribute, particularly in EMT. The whole-step and task clocks cover different boundaries, so the remainder is an approximate subtraction. The profiler itself adds overhead; use the separate unprofiled controls for absolute performance claims. The controls are single-run comparisons, so small differences can reflect normal run-to-run variation.

The CSV files alongside this notebook preserve every task timing and every measured step for further inspection.